# Chapter 43: Bayesian, causal and text mechanisms

Fresh small calculations cover conjugate updating, fixed-hyperparameter pooling, Simpson standardization, collider simulation, PIT filtering, embeddings, negation and prompt hashing. Text prediction estimates and five PNGs are frozen accepted assets. No TF-IDF/Ridge retraining or LLM call occurs. The existing T43-01/T43-02 display repairs remain closed; this is not new market evidence.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). This stage delivers language views; portable dependencies and execution are checked in the later isolated cold start.

## 0. Shared modules and frozen assets

In [ ]:
import os, sys
from pathlib import Path
from uuid import uuid4
candidate = Path.cwd().resolve()
ROOT = next((p for p in (candidate, *candidate.parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start from the reader package root or a subdirectory")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("ch43_bayesian_text_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
workspace_root = ROOT
print("Run directory:", RUN_DIR.relative_to(ROOT))
import numpy as np, pandas as pd
from IPython.display import Image, display
import json, hashlib
from quant_course.bayesian_text import (
    beta_binomial_posterior_predictive,
    normal_hierarchical_partial_pooling,
    generate_causal_dataset,
    compute_stratified_ate,
    demonstrate_collider_bias,
    generate_constructed_text_dataset,
    filter_pit_eligible_documents,
    hand_embedding_matrix,
    negation_sentiment_scorer,
    llm_offline_audit_ledger,
    downstream_execution_interface_spec,
)

CANDIDATE = ROOT / 'fixtures/late'
AUDIT = json.loads((CANDIDATE / 'display_audit.json').read_text())


## 1. Bayesian updating

### 1.1. Beta–Binomial

Prior Beta(2,2) and eight successes in ten trials give Beta(10,4), mean10/14, variance $2/147$ and joint two-success predictive probability $11/21$. Future successes are conditionally independent given p, but dependent after integrating over posterior uncertainty: predictive covariance equals posterior Var(p).

In [ ]:
bayes_res = beta_binomial_posterior_predictive(alpha_prior=2.0, beta_prior=2.0, successes=8, trials=10)
print(f"Alpha Posterior: {bayes_res['alpha_post']}")
print(f"Beta Posterior: {bayes_res['beta_post']}")
print(f"Posterior Mean: {bayes_res['posterior_mean']:.4f}")
print(f"Posterior Variance: {bayes_res['posterior_variance']:.6f}")
print(f"Joint 2 Successes Predictive: {bayes_res['joint_two_successes']:.4f}")
print(f"Predictive Covariance: {bayes_res['predictive_covariance']:.6f}")

### 1.2. Normal partial pooling

With prior mean 0, between-group SD 0.02 and observation SD 0.04, groups (n=4, mean=0.03) and (n=36, mean=0.01) receive weights 0.5 and 0.9. Posterior means are 0.015 and 0.009. Hyperparameters are specified, not learned here.

In [ ]:
groups_spec = [{'n': 4, 'sample_mean': 0.03}, {'n': 36, 'sample_mean': 0.01}]
normal_res = normal_hierarchical_partial_pooling(
    prior_mean=0.0, prior_sd=0.02, observation_sd=0.04, groups=groups_spec
)
for g in normal_res['groups']:
    print(f"Group {g['group_id']}: n={g['n']}, sample_mean={g['sample_mean']:.3f}, "
          f"shrinkage_weight={g['shrinkage_weight']:.2f}, post_mean={g['posterior_mean']:.4f}, "
          f"post_var={g['posterior_var']:.6f}, pred_var={g['predictive_var']:.6f}")

### 1.3. Frozen Bayesian figure

Display the accepted PNG, without regenerating it from the small calculations.

In [ ]:
fig1_path = CANDIDATE / 'ch43' / 'fig43_1_bayesian_updating.png'
display(Image(filename=str(fig1_path)))

## 2. Causal inference

### 2.1. Simpson reversal and standardization

The constructed attention strata give an unadjusted +0.38 probability contrast but −0.10 within each stratum. Standardization to equal stratum weights gives −0.10. Interpreting it as ATE requires consistency, exchangeability and positivity; arithmetic adjustment alone cannot establish those assumptions in observed market data.

In [ ]:
causal_df = generate_causal_dataset(seed=4301)
causal_res = compute_stratified_ate(
    causal_df, z_col='Z', t_col='T', y_col='Y', target_p_z={'high': 0.5, 'low': 0.5}
)
print(f"Unadjusted Difference: {causal_res['unadjusted_difference']:+.2f}")
print(f"High Stratum Contrast: {causal_res['stratum_stats']['high']['contrast']:+.2f}")
print(f"Low Stratum Contrast:  {causal_res['stratum_stats']['low']['contrast']:+.2f}")
print(f"Standardized ATE:      {causal_res['standardized_ate']:+.2f}")
print(f"Simpson Reversal Detected: {causal_res['simpson_reversal']}")

### 2.2. Collider selection

Generate 2,000 observations from two independent Bernoulli indicators and select $C=1\{T=1\lor Y=1\}$. Conditional covariance approaches −1/9; the finite simulation need not equal it exactly. Conditioning on the common effect creates association.

In [ ]:
collider_res = demonstrate_collider_bias(n=2000, seed=4301)
print(f"Unconditioned Covariance: {collider_res['unconditioned_covariance']:+.4f}")
print(f"Conditioned on Collider Covariance: {collider_res['conditioned_on_collider_covariance']:+.4f}")
print(f"Theoretical Covariance: {-1.0/9.0:+.4f}")

### 2.3. Frozen Simpson and DAG figures

Display two accepted images; no new causal figure is written.

In [ ]:
fig2_path = CANDIDATE / 'ch43' / 'fig43_2_causal_simpson.png'
display(Image(filename=str(fig2_path)))

fig5_path = CANDIDATE / 'ch43' / 'fig43_5_causal_dag.png'
display(Image(filename=str(fig5_path)))

## 3. PIT text contract

### 3.1. Constructed 120-event fixture

Read the frozen CSV, parse its UTC fields and filter by signal time. Its four fixed templates repeat through 59/29/30 mature training/validation/test blocks. The constructed New York sequence has release 16:05, vendor availability 16:25 and ingestion 16:30, preceding next-day 09:00 signals. This tests extrapolation across time blocks, not generalization to unseen prose.

In [ ]:
base_text_df = pd.read_csv(CANDIDATE/'ch43/constructed_documents.csv')
for name in ['event_time','release_time','available_at','ingested_at','revision_time','signal_time','trade_time','label_start','label_end','label_available_at']:
 if name in base_text_df:base_text_df[name]=pd.to_datetime(base_text_df[name],utc=True)
eligible_df=filter_pit_eligible_documents(base_text_df,cutoff_col='signal_time')
assert len(eligible_df)==120
print(AUDIT['t43_01'])

### 3.2. Late-revision negative control

Add a version available and ingested after its signal. Filtering keeps 120 eligible documents and retains the earlier version of document 10. No model is fitted.

In [ ]:
late_rev=base_text_df.iloc[10:11].copy()
late_rev['version_id']=2
late_rev['revision_time']=late_rev.signal_time+pd.Timedelta(hours=1)
late_rev['available_at']=late_rev.revision_time
late_rev['ingested_at']=late_rev.available_at+pd.Timedelta(minutes=5)
filtered=filter_pit_eligible_documents(pd.concat([base_text_df,late_rev],ignore_index=True),cutoff_col='signal_time')
assert len(filtered)==120
assert int(filtered.loc[filtered.doc_id==10,'version_id'].iloc[0])==1
print('Late-revision small negative control retained; no fit.')

### 3.3. Frozen TF-IDF/Ridge estimates

Read the accepted text summary with 59/29/30 mature blocks. Historical comparison used a training-mean baseline and validation-selected Ridge. Test MSE/R² and Spearman correlation aggregate 30 test dates; one asset per day leaves cross-sectional IC undefined, and the constant baseline's rank correlation is undefined. Frozen vocabulary/IDF or an OOV check does not prove new-text ability.

In [ ]:
text_res=json.loads((CANDIDATE/'ch43/ch43_summary.json').read_text())['text_nlp']
display(pd.Series({k:v for k,v in text_res.items() if isinstance(v,(str,int,float))}))
assert text_res['n_train_eligible']==59 and text_res['n_val_eligible']==29 and text_res['n_test']==30
print('Frozen prediction estimates, not fitted in this revision.')

### 3.4. Frozen timeline and metrics

Display two accepted PNGs. Along with the three earlier images, these make five frozen displays, with no newly saved plot.

In [ ]:
fig3_path = CANDIDATE / 'ch43' / 'fig43_3_pit_text_timeline.png'
display(Image(filename=str(fig3_path)))

fig4_path = CANDIDATE / 'ch43' / 'fig43_4_text_prediction_metrics.png'
display(Image(filename=str(fig4_path)))

## 4. Embeddings and negation

### 4.1. A 6×3 teaching matrix

Compute normalized vectors, cosine similarities and pooling without downloading an external embedding model. These stipulated vectors illustrate geometry rather than learned semantic performance.

In [ ]:
df_emb, sims = hand_embedding_matrix()
print("Normalized Embedding Matrix:")
print(df_emb.round(3))
print("\nCosine Similarities:")
for k, v in sims.items():
    print(f"{k}: {v:+.4f}")

### 4.2. Negation window

Compare a naive unigram score with a negation-aware score for the given English sentence. The polarity reversal is a small rule example, not broad sentiment validation.

In [ ]:
sentence = 'earnings performance did not improve showing no growth despite aggressive restructuring'
score_res = negation_sentiment_scorer(sentence)
print(f"Input Text: {score_res['text']}")
print(f"Naive Unigram Score:       {score_res['naive_unigram_score']} (misclassifies positive words)")
print(f"Negation-Aware Score:      {score_res['negation_aware_score']} (correctly flips polarity)")
print(f"Polarity Flip Detected:    {score_res['polarities_differ']}")

## 5. Offline prompt audit

### 5.1. Full-input SHA-256

Model status stays not_run and its real cutoff is unknown; the stated 2023-12-31 cutoff is hypothetical. Hash the full prompt including input text. Recomputing after changing both input and local digest can still pass local consistency; detecting alteration requires an independently trusted prior digest. Hashes are neither encryption nor immutable timestamped custody. Temperature zero and a seed do not ensure cross-platform reproducibility.

In [ ]:
ledger = llm_offline_audit_ledger()
print(f"Prompt Template Version: {ledger['prompt_template_version']}")
print(f"Full Prompt SHA-256:     {ledger['full_prompt_sha256_hash']}")
print(f"Model Status:            {ledger['model_status']}")
print(f"Declared Model Cutoff:   {ledger['declared_model_cutoff']}")
print(f"Provenance Table:        {ledger['provenance_table']}")
print("Reproducibility Limits:")
for lim in ledger['reproducibility_limits']:
    print(f" - {lim}")
# Assume the baseline digest was independently retained before local mutation.
from quant_course.bayesian_text import compute_prompt_hash, OFFLINE_PROMPT_TEMPLATE
trusted_digest=ledger['full_prompt_sha256_hash']
changed=llm_offline_audit_ledger(sample_text='changed input and locally matching hash')
assert compute_prompt_hash(OFFLINE_PROMPT_TEMPLATE,changed['sample_input_text'])==changed['full_prompt_sha256_hash']
assert changed['full_prompt_sha256_hash']!=trusted_digest
print('Local consistency passes after input+digest mutation; independent baseline differs.')


### 5.2. Prediction-only downstream contract

Display the schema for passing a time-qualified prediction to a downstream holdings/fee ledger. No rotating-asset fills or trading ledger is fabricated. No API call, fee or external data transfer occurs.

In [ ]:
downstream_spec = downstream_execution_interface_spec()
print(f"Mode: {downstream_spec['mode']}")
print(f"Required Schema: {downstream_spec['required_schema']}")
print(f"Scope Boundary: {downstream_spec['scope_boundary']}")